# AI Study Buddy · Phase 10 · Fine-tune Qwen3-1.7B on SciQ (Colab T4)

This notebook trains the app's **"finetuned"** answer generator. Run the cells **top to bottom**.

| Step | What happens | Saved to Google Drive (`MyDrive/study-buddy-finetune/`) |
|---|---|---|
| 1–3 | GPU check, Drive, install pinned libraries | – |
| 4 | Build 3,000 train / 300 validation / 500 test examples from SciQ | `data/`, `results/data_stats.json` |
| 5–6 | 4-bit QLoRA fine-tuning, 1 epoch, fp16 | `checkpoints/` (resumable), `results/training.json` |
| 7 | Export: LoRA adapter, merged model, GGUF q4_k_m | `adapter/`, `merged/`, `gguf/` |
| 8 | Base vs fine-tuned on the same 500 test items | `results/eval_hf.json`, `results/predictions_hf_*.jsonl` |
| 9 | (Optional) push to the Hugging Face Hub | – |

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Put the repo's `finetune/` folder in Drive (see step 2) or set `REPO_URL`.

Every number in the project's report must come from the files in `results/`. Nothing is typed by hand.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| torch", torch.__version__, "| bf16 supported:", torch.cuda.is_bf16_supported())

## 2. Google Drive and the project files

Checkpoints and outputs go to Drive, so a disconnected session can resume.

The code comes from **one** of:
- **A.** `REPO_URL`: a git URL of this project (public repo, or a URL with a token you trust Colab with), or
- **B.** the files of the repo's `finetune/` folder in `MyDrive/study-buddy-finetune/code/`, i.e.
  `code/prepare_data.py` (a nested `code/finetune/prepare_data.py` also works). Leave `REPO_URL` empty.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, sys
DRIVE = "/content/drive/MyDrive/study-buddy-finetune"
for sub in ("data", "results", "checkpoints", "adapter", "merged", "gguf"):
    os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)

REPO_URL = ""  # option A, e.g. "https://github.com/<you>/AI-study-buddy.git"
if REPO_URL:
    if not os.path.isdir("/content/repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
    FT_DIR = "/content/repo/finetune"
else:  # option B: accepts code/prepare_data.py or code/finetune/prepare_data.py
    src = f"{DRIVE}/code"
    if not os.path.exists(f"{src}/prepare_data.py") and os.path.exists(f"{src}/finetune/prepare_data.py"):
        src = f"{src}/finetune"
    FT_DIR = "/content/finetune"
    shutil.copytree(src, FT_DIR, dirs_exist_ok=True)

for f in ("prepare_data.py", "prompts.py", "metrics.py", "eval_runner.py", "evaluate.py", "set_chat_template.py",
          "requirements-colab.txt", "MODEL_CARD.md"):
    assert os.path.exists(f"{FT_DIR}/{f}"), f"missing {f} in {FT_DIR}"
# append (not insert): the project's evaluate.py must never shadow the `evaluate` pip package
if FT_DIR not in sys.path:
    sys.path.append(FT_DIR)
print("code:", FT_DIR, "| outputs:", DRIVE)

## 3. Install the pinned libraries

Two steps (explained at the top of `requirements-colab.txt`). Colab's own torch stays as it is.

In [ ]:
import re, subprocess, sys, torch
TORCH = torch.__version__.split("+")[0]
text = open(f"{FT_DIR}/requirements-colab.txt").read()
no_deps = re.findall(r"^([A-Za-z0-9_.\-]+==[^\s#]+)", text.split("# [deps]")[0].split("# [no-deps]")[1], re.M)
deps = re.findall(r"^([A-Za-z0-9_.\-]+==[^\s#]+)", text.split("# [deps]")[1], re.M)
print("no-deps:", no_deps); print("deps:", deps)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *no_deps], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps, f"torch=={TORCH}"], check=True)

In [ ]:
# Record every version that matters (written into results/training.json later).
import importlib.metadata as md, platform
VERSIONS = {"python": platform.python_version()}
for pkg in ("torch", "torchvision", "triton", "unsloth", "unsloth_zoo", "transformers", "tokenizers", "trl", "peft",
            "accelerate", "bitsandbytes", "datasets", "huggingface_hub", "torchao", "xformers"):
    try:
        VERSIONS[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        VERSIONS[pkg] = None
VERSIONS

## 4. Build the datasets

`prepare_data.py` downloads SciQ at a pinned revision, drops rows without a support paragraph and writes
the three JSONL files plus `results/data_stats.json` (all counts come from this run).

In [ ]:
subprocess.run([sys.executable, "prepare_data.py", "--out", f"{DRIVE}/data", "--stats", f"{DRIVE}/results/data_stats.json"],
               cwd=FT_DIR, check=True)
import json
DATA_STATS = json.load(open(f"{DRIVE}/results/data_stats.json"))
{s: {k: v for k, v in d.items() if k in ("rows", "dropped_empty_support", "examples", "by_task", "tokens")}
 for s, d in DATA_STATS["splits"].items()}

## 5. Load Qwen3-1.7B in 4-bit and add LoRA

T4 has no bfloat16, so everything runs in **fp16**. Settings from IMPLEMENTATION_PLAN §8.3.

In [ ]:
from unsloth import FastLanguageModel  # import unsloth before transformers/trl
import torch
from prepare_data import BASE_MODEL, BASE_MODEL_REVISION, MAX_SEQ_LEN, SEED

CONFIG = {
    "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
    "load_in_4bit": True, "dtype": "float16", "max_seq_len": MAX_SEQ_LEN,
    "lora": {"r": 16, "alpha": 16, "dropout": 0.0,
             "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]},
    "train": {"epochs": 1, "lr": 2e-4, "scheduler": "cosine", "warmup_steps": 10, "per_device_batch": 4,
              "grad_accum": 4, "effective_batch": 16, "optim": "adamw_8bit", "weight_decay": 0.01,
              "fp16": True, "bf16": False, "eval_steps": 25, "save_steps": 50, "seed": SEED},
    "enable_thinking": False,
    "train_on": "assistant tokens only (after the empty think block)",
}
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, revision=BASE_MODEL_REVISION, max_seq_length=MAX_SEQ_LEN,
    dtype=torch.float16, load_in_4bit=True,
    # Without this Unsloth swaps in its pre-quantised "-unsloth-bnb-4bit" repo and ignores the pinned revision.
    use_exact_model_name=True,
)
LOADED_FROM = getattr(model.config, "_name_or_path", "")
assert "bnb-4bit" not in LOADED_FROM, f"loaded {LOADED_FROM}, not the pinned {BASE_MODEL}"
CONFIG["loaded_from"] = LOADED_FROM
print("loaded:", LOADED_FROM, "@", BASE_MODEL_REVISION)
model = FastLanguageModel.get_peft_model(
    model, r=CONFIG["lora"]["r"], lora_alpha=CONFIG["lora"]["alpha"], lora_dropout=CONFIG["lora"]["dropout"],
    target_modules=CONFIG["lora"]["target_modules"], bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
model.print_trainable_parameters()

## 6. Train (1 epoch, checkpoints in Drive)

Each example is rendered with Qwen3's chat template and `enable_thinking=False`. Only the assistant's
answer is trained (the prompt and the empty think block are masked). If the session dies, run steps
1–5 again and this cell **resumes from the last checkpoint** in Drive.

In [ ]:
import json, time, datetime as dt
from datasets import load_dataset
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from prompts import INSTRUCTION_MARKER, RESPONSE_MARKER

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, enable_thinking=False)}

raw = load_dataset("json", data_files={"train": f"{DRIVE}/data/train.jsonl", "validation": f"{DRIVE}/data/val.jsonl"})
ds = raw.map(to_text, remove_columns=raw["train"].column_names)
assert RESPONSE_MARKER in ds["train"][0]["text"], "thinking-off template marker missing"

t = CONFIG["train"]
args = SFTConfig(
    output_dir=f"{DRIVE}/checkpoints", dataset_text_field="text", max_length=MAX_SEQ_LEN, packing=False,
    num_train_epochs=t["epochs"], learning_rate=t["lr"], lr_scheduler_type=t["scheduler"], warmup_steps=t["warmup_steps"],
    per_device_train_batch_size=t["per_device_batch"], per_device_eval_batch_size=t["per_device_batch"],
    gradient_accumulation_steps=t["grad_accum"], optim=t["optim"], weight_decay=t["weight_decay"],
    fp16=True, bf16=False, logging_steps=5, eval_strategy="steps", eval_steps=t["eval_steps"],
    save_strategy="steps", save_steps=t["save_steps"], save_total_limit=3, seed=t["seed"], report_to="none",
)
trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds["train"], eval_dataset=ds["validation"], args=args)
trainer = train_on_responses_only(trainer, instruction_part=INSTRUCTION_MARKER, response_part=RESPONSE_MARKER)

# Check the masking on one example: the trained tokens must be exactly the JSON answer.
lab = [x for x in trainer.train_dataset[0]["labels"] if x != -100]
trained = tokenizer.decode(lab)
print("trained tokens of example 0:", trained[:200])
assert trained.lstrip().startswith("{"), "masking is wrong: trained text should start with the JSON answer"

import os
SESSIONS_PATH = f"{DRIVE}/results/training_sessions.jsonl"  # one line per trainer.train() call, never overwritten
last = get_last_checkpoint(args.output_dir)
if last is None and os.path.exists(SESSIONS_PATH):
    os.remove(SESSIONS_PATH)  # fresh run (no checkpoints): earlier sessions belong to another model
steps_before = int(last.rsplit("-", 1)[1]) if last else 0
print("resuming from:", last)
started = dt.datetime.now(dt.UTC)
t0 = time.perf_counter()

def log_session(steps_to, train_runtime_s=None, done=False):
    # One line per checkpoint save and one at the end, so a disconnect keeps the time up to the last checkpoint.
    rec = {"started_at": started.isoformat(timespec="seconds"), "resumed_from_checkpoint": last,
           "steps_from": steps_before, "steps_to": steps_to, "wall_time_s": round(time.perf_counter() - t0, 1),
           "train_runtime_s": train_runtime_s, "done": done}
    with open(SESSIONS_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n")

from transformers import TrainerCallback
class SessionLog(TrainerCallback):
    def on_save(self, args, state, control, **kw):
        log_session(state.global_step)
trainer.add_callback(SessionLog())

train_out = trainer.train(resume_from_checkpoint=last)
TRAIN_WALL_S = time.perf_counter() - t0
log_session(trainer.state.global_step, train_out.metrics.get("train_runtime"), done=True)
final_eval = trainer.evaluate()
print(train_out.metrics, final_eval)

In [ ]:
# results/training.json: real values only. Training time = sum over the sessions that trained steps,
# so a resume after a disconnect adds up, and re-running this cell after training is harmless.
SESSIONS = {}
for line in open(SESSIONS_PATH):
    if line.strip():
        rec = json.loads(line)
        SESSIONS[rec["started_at"]] = rec  # the latest line of each session wins
SESSIONS = list(SESSIONS.values())
trained_sessions = [s for s in SESSIONS if s["steps_to"] > s["steps_from"]]
covered = sorted((s["steps_from"], s["steps_to"]) for s in trained_sessions)
complete = bool(covered) and covered[0][0] == 0 and covered[-1][1] == trainer.state.global_step and all(
    a[1] >= b[0] for a, b in zip(covered, covered[1:]))
TRAINING = {
    "created_at": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
    "gpu": GPU_NAME, "versions": VERSIONS, "config": CONFIG,
    "dataset": {"name": DATA_STATS["dataset"], "revision": DATA_STATS["dataset_revision"],
                "train_examples": len(ds["train"]), "validation_examples": len(ds["validation"])},
    "sessions": SESSIONS,                         # every trainer.train() call (incl. no-op resumes)
    # wall clock of trainer.train() summed over sessions; an interrupted session counts up to its last checkpoint
    "train_wall_time_s_total": round(sum(s["wall_time_s"] for s in trained_sessions), 1),
    "train_time_complete": complete,              # False = some steps' time is missing (e.g. an older notebook)
    "last_train_metrics": train_out.metrics,      # of this cell run only
    "final_eval": final_eval,
    "global_steps": trainer.state.global_step,
    "log_history": trainer.state.log_history,    # train loss every 5 steps, eval loss every 25 (kept across resumes)
}
json.dump(TRAINING, open(f"{DRIVE}/results/training.json", "w"), indent=2, default=str)
print("saved results/training.json | train time (s):", TRAINING["train_wall_time_s_total"], "| complete:", complete)

## 7. Export: LoRA adapter → merged model → GGUF q4_k_m

The GGUF step builds llama.cpp inside Colab the first time (several minutes). The final GGUF gets the
thinking-off chat template embedded (`set_chat_template.py`), because Ollama uses the template inside the GGUF.

In [ ]:
import glob, os, time
EXPORTS = {}
t0 = time.perf_counter()
model.save_pretrained(f"{DRIVE}/adapter"); tokenizer.save_pretrained(f"{DRIVE}/adapter")
EXPORTS["adapter_s"] = round(time.perf_counter() - t0, 1)

t0 = time.perf_counter()
model.save_pretrained_merged(f"{DRIVE}/merged", tokenizer, save_method="merged_16bit")
EXPORTS["merged_s"] = round(time.perf_counter() - t0, 1)

t0 = time.perf_counter()
GGUF_TMP = "/content/gguf_out"
model.save_pretrained_gguf(GGUF_TMP, tokenizer, quantization_method="q4_k_m")
EXPORTS["gguf_s"] = round(time.perf_counter() - t0, 1)
found = [p for p in glob.glob(f"{GGUF_TMP}*/**/*.gguf", recursive=True) + glob.glob(f"{GGUF_TMP}*.gguf")
         if "q4_k_m" in p.lower()]
assert found, "no q4_k_m GGUF found; look at the output above"
# Embed the thinking-off chat template (Ollama renders Qwen3 with the GGUF's own template; see Modelfile).
GGUF_PATH = f"{DRIVE}/gguf/study-buddy-qwen-q4_k_m.gguf"
subprocess.run([sys.executable, "set_chat_template.py", found[0], GGUF_PATH], cwd=FT_DIR, check=True)
EXPORTS["gguf_file"] = os.path.basename(GGUF_PATH)
EXPORTS["gguf_bytes"] = os.path.getsize(GGUF_PATH)
TRAINING["exports"] = EXPORTS
json.dump(TRAINING, open(f"{DRIVE}/results/training.json", "w"), indent=2, default=str)
EXPORTS

## 8. Evaluate: base vs fine-tuned on the same 500 test items

One 4-bit model: **base** = LoRA adapter switched off, **fine-tuned** = adapter on. Same prompts
(thinking off), greedy decoding, same token limit. Writes `results/eval_hf.json` and the raw predictions.
Takes a while (1,000 generations).

In [ ]:
from eval_runner import HFBackend, load_items
from evaluate import run_comparison
FastLanguageModel.for_inference(model)
items = load_items(f"{DRIVE}/data/test_eval.jsonl")
report = run_comparison(
    HFBackend(model, tokenizer, batch_size=8), items, {"base": "base", "finetuned": "finetuned"},
    f"{DRIVE}/results", "hf",
    extra={"gpu": GPU_NAME, "versions": VERSIONS, "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
           "latency_note": "GPU time per item, batched (batch time / batch size), 4-bit weights", "json_mode": False},
)
{label: {k: v for k, v in m.items() if k in ("gqa", "refusal", "mcq", "latency_ms")} for label, m in report["models"].items()}

## 9. (Optional) Push to the Hugging Face Hub

Add a token named `HF_TOKEN` in Colab's **Secrets** panel (key icon, "Notebook access" on). Nothing is
pushed unless you set `HF_REPO`. Keep the repo private unless you want it public; SciQ is CC-BY-NC-3.0,
so the fine-tuned model is for non-commercial use.

In [ ]:
HF_REPO = ""          # e.g. "<your-username>/study-buddy-qwen3-1.7b-sciq"
HF_PRIVATE = True
if HF_REPO:
    from google.colab import userdata
    from huggingface_hub import HfApi
    token = userdata.get("HF_TOKEN")
    api = HfApi(token=token)
    api.create_repo(HF_REPO, private=HF_PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=f"{DRIVE}/adapter", repo_id=HF_REPO, path_in_repo="adapter")
    api.upload_file(path_or_fileobj=GGUF_PATH, path_in_repo=os.path.basename(GGUF_PATH), repo_id=HF_REPO)
    api.upload_file(path_or_fileobj=f"{FT_DIR}/MODEL_CARD.md", path_in_repo="README.md", repo_id=HF_REPO)
    print("pushed to https://huggingface.co/" + HF_REPO)
else:
    print("HF_REPO is empty: nothing pushed")

## 10. Bring the results home

Download from Drive (`MyDrive/study-buddy-finetune/`):
- `results/*.json` and `results/*.jsonl` → the repo's `finetune/results/`
- `gguf/study-buddy-qwen-q4_k_m.gguf` → the repo's `finetune/` (gitignored), then follow **finetune/README.md → "Use it in the app"**.